# C4 — Credit index, tranches and index options

**Audience:** analysts who can price a single-name CDS and interpret hazard curves.

**Outcome:** distinguish index protection, synthetic tranches, and payer option front-end protection; reconcile the option to its knockout form and preserve calibrated quote risk. This lab extends the common book independently of the volatility track. Every quote is synthetic and fixed as of 2025-01-15.

## Financial context and interpretation

### Index identity and state are separate inputs

An index's name, series and version identify its contract basket. Its fixed running coupon is not its current fair spread. This synthetic 125-name index uses a 100 bp coupon and a 120 bp calibration quote. Positive protection-buyer value therefore combines the protection leg and the off-market premium leg; it cannot be inferred by multiplying the spread difference by maturity.

Series identifies the periodic basket and maturity convention; version tracks subsequent constituent events. Index factor measures surviving notional relative to the original amount. Our controlled comparison prices a single calibrated curve and then 125 equally weighted, identical constituent curves. Their prices agree because their economics agree, not because actual constituents are homogeneous. The next version marks one constituent defaulted and reduces the factor. Settled default loss and the value of surviving protection are reported separately.

Recovery links an approximate flat hazard to spread through `hazard ≈ spread/(1-recovery)`. This is an intuition check. The native bootstrap instead observes the premium schedule, accrued premium on default, discounting and protection timing. Its replay recipe is retained for quote-space CS01. A sensitivity without the original quote units and calibration convention is not an auditable hedge input.

### Options are on forward protection economics

A payer buys the right to pay a fixed spread and receive protection; it benefits from spread widening. A receiver takes the opposite option direction. The native `bloomberg_cdso` selector uses a spread-option treatment with risky-annuity and front-end-protection conventions. A knockout option can terminate on the relevant default event. Non-knockout index treatment can retain protection for losses before option expiry. Always state factor, realized index loss, settlement and knockout conventions together.

The spread strike is decimal: `0.01` means 100 bp. The example surface contains lognormal spread volatility, also decimal: `0.40` means 40%. This is not a 40 bp normal volatility. Surface expiry and strike axes must match the instrument's lookup convention. `implied_vol` below inverts the model's own value and is a consistency check; it is not an independent calibration to an observed premium. `par_spread` metrics are displayed in basis points, so they must not be inserted unconverted into a decimal strike field.

### Tranche delta and correlation are different risks

The synthetic 0–3% index tranche divides portfolio losses by attachment/detachment points. Base-correlation nodes use percentage attachment coordinates. Correlation changes the loss distribution even at the same expected portfolio loss, and therefore redistributes risk across equity and senior tranches. The sweep below holds hazard curves and discounting fixed. Its declining equity-protection value is specific to that tranche and setup; do not extend the sign to every tranche.

Tranche delta is calculated by repricing both the tranche and the index along the same one-basis-point quote direction. It is a local ratio of value changes, not a universal hedge multiplier. The actual BBB holding in this program is a cash CLO tranche with a waterfall. Its native `cs01` is a z-spread discount sensitivity, whereas index `cs01` replays market CDS quotes. A formal match of these dollar numbers leaves both model-coordinate basis and economic basis. Joint scenario repricing is mandatory.

In [ ]:
from pathlib import Path
import sys
from copy import deepcopy
from datetime import date
import json
import math
import numpy as np
import pandas as pd
from _shared import analyst_tracks as tracks
from finstack_quant.valuations.instruments import price_instrument
AS_OF = date(2025, 1, 15)


## 1. Build the index from contracts and quotes

The index contains 125 names and carries a 100 bp contractual running coupon. Its calibration uses a 120 bp par-spread curve. The mark is therefore a protection-leg minus premium-leg value, not a statement that the coupon equals the par spread. The simple approximation $h\approx s/(1-R)$ is a useful unit check, not a replacement for the dated CDS calibration.

In [ ]:
from finstack_quant.calibration import calibrate
inputs = tracks.credit_index_inputs()
market = tracks.build_market("credit")
calibration = calibrate(tracks.credit_calibration_envelope())
index = price_instrument(json.dumps(inputs["ANALYST-CDX"]), market, AS_OF, model="hazard_rate")
hazard_approx = 120 / 10000 / (1 - 0.4)
print(pd.DataFrame([{"names": 125, "running_coupon_bp": 100, "calibration_par_bp": 120,
               "hazard_approx_decimal": hazard_approx, "index_pv_usd": index.value.amount}]))

## 2. Price a payer option and isolate front-end protection

A payer option gives the right to buy protection. For this non-knockout index payer, defaults before expiry contribute front-end protection. A knockout option excludes that contribution. Keep strike, maturity, recovery, index factor and the spread-volatility surface identical before comparing the two. Surface strikes are decimal spreads: 0.01 means 100 bp.

In [ ]:
payer = inputs["CDX-PAYER"]
knockout = deepcopy(payer)
knockout["instrument"]["spec"]["knockout"] = True
nonko = price_instrument(json.dumps(payer), market, AS_OF, model="bloomberg_cdso", metrics=["cs01", "bucketed_cs01"])
ko = price_instrument(json.dumps(knockout), market, AS_OF, model="bloomberg_cdso")
fep_increment = nonko.value.amount - ko.value.amount
assert nonko.value.amount > ko.value.amount > 0
print(pd.DataFrame({"case": ["non-knockout payer", "knockout payer", "front-end increment"],
              "usd": [nonko.value.amount, ko.value.amount, fep_increment]}))

The positive difference is an invariant of this payer fixture, not a universal statement about every receiver, strike convention or realized-loss treatment. The native index factor and realized loss fields carry trade state; reducing notional silently is not an equivalent lifecycle model.

In [ ]:
risk = pd.Series(nonko.metrics, name="usd_per_1bp").filter(like="cs01::")
assert any(key.startswith("cs01::CDX-HAZ") for key in nonko.metrics)
print(risk.to_frame())

## 3. Contrast synthetic index tranches with a cash CLO

The 0–3% synthetic tranche transfers a slice of index default loss. Its base-correlation input is a decimal correlation keyed by detachment percentage points. This is distinct from the real cashflow CLO and its BBB sleeve in C3. A CDS-tranche proxy does not establish CLO waterfall behavior.

In [ ]:
from finstack_quant.core.market_data import BaseCorrelationCurve
tranche_rows = []
for rho in [0.15, 0.25, 0.40]:
    scenario_market = tracks.build_market("credit")
    scenario_market.insert(BaseCorrelationCurve("CDX-BASE-CORR", [(3.0, rho), (7.0, rho), (15.0, rho), (100.0, rho)]))
    result = price_instrument(json.dumps(inputs["CDX-0-3"]), scenario_market, AS_OF, model="hazard_rate")
    tranche_rows.append({"base_correlation": rho, "0_3_protection_pv_usd": result.value.amount})
print(pd.DataFrame(tranche_rows))

## Exercise — Move strike and explain the position

Raise the payer strike from 100 to 150 bp, leaving the same calibrated hazard curve. Predict the premium change before running the cell. Explain why this option can hedge spread widening but leaves index-basis, recovery and jump-to-default risk.

In [ ]:
higher_strike = deepcopy(payer)
higher_strike["instrument"]["spec"]["strike"] = {"spread": "0.015"}
higher = price_instrument(json.dumps(higher_strike), market, AS_OF, model="bloomberg_cdso").value.amount
assert higher < nonko.value.amount
print(pd.Series({"100bp_strike_usd": nonko.value.amount, "150bp_strike_usd": higher}))

**Review checkpoint:** the index uses `hazard_rate`, the option uses `bloomberg_cdso`, and native quote-space CS01 comes from a lossless calibration recipe. A directly authored flat hazard curve would support intensity risk, not the same quote-risk claim.

### Reconcile a single-curve index with constituent pricing

In [ ]:
from finstack_quant.valuations.instruments import validate_instrument_json
constituent_index=deepcopy(inputs["ANALYST-CDX"])
s=constituent_index["instrument"]["spec"]
s["pricing"]="constituents"
s["constituents"]=[{"credit":{"credit_curve_id":"CDX-HAZ","recovery_rate":.4,"reference_entity":"SYNTHETIC-CONSTITUENT"},
                     "weight":1/125,"defaulted":False} for _ in range(125)]
validate_instrument_json(json.dumps(constituent_index))
by_name=price_instrument(json.dumps(constituent_index),market,AS_OF,model="hazard_rate",metrics=["par_spread"])
assert abs(by_name.value.amount-index.value.amount)<.01
post_default=deepcopy(constituent_index)
p=post_default["instrument"]["spec"];p["version"]=2;p["index_factor"]=124/125;p["constituents"][-1]["defaulted"]=True
survivors=price_instrument(json.dumps(post_default),market,AS_OF,model="hazard_rate",metrics=["par_spread"])
assert abs(survivors.value.amount-by_name.value.amount*124/125)<.01
assert abs(survivors.metrics["par_spread"]-by_name.metrics["par_spread"])<1e-8
print({"single_curve_pv":index.value.amount,"constituent_pv":by_name.value.amount,
       "survivor_pv":survivors.value.amount,"settled_default_cash_usd":1_000_000*.6/125,
       "par_spread_before":by_name.metrics["par_spread"],"par_spread_after":survivors.metrics["par_spread"]})

### Credit-option volatility, Greeks and default semantics

In [ ]:
option_risk=price_instrument(json.dumps(payer),market,AS_OF,model="bloomberg_cdso",
    metrics=["implied_vol","vega","spread_dv01","par_spread"])
assert option_risk.metrics["implied_vol"]>0
receiver=deepcopy(payer);receiver["instrument"]["spec"]["option_type"]="put"
receiver_price=price_instrument(json.dumps(receiver),market,AS_OF,model="bloomberg_cdso").value.amount
post_option=deepcopy(payer)
post_option["instrument"]["spec"].update({"index_factor":124/125,"realized_index_loss":.6/125})
post_option_price=price_instrument(json.dumps(post_option),market,AS_OF,model="bloomberg_cdso").value.amount
assert receiver_price>=0 and post_option_price>=0
print(pd.Series(option_risk.metrics))
print({"payer_pv":nonko.value.amount,"receiver_pv":receiver_price,"post_default_payer_pv":post_option_price})
print("Implied volatility here inverts the model PV; it is a consistency diagnostic, not an independent market calibration.")

### Exercise — Match a sensitivity, then measure the residual

Size index protection against the BBB sleeve's reported CS01. Reprice a 100 bp index widening with 5% annual collateral defaults, 25% recovery and a six-month recovery lag.

The formal CS01 match is exact in the displayed arithmetic. It deliberately cannot establish hedge completeness because the sensitivities use different coordinates. The stress reprices the actual BBB waterfall and the index separately, then reconciles their combined P&amp;L. The residual is the quantity to discuss with the PM; it includes the chosen mapping between an index shock and collateral deterioration.

In [ ]:
from finstack_quant.valuations.instruments import structured_credit_tranche_metrics
holding=tracks.clo_bbb_holding()
bbb=structured_credit_tranche_metrics(json.dumps(holding["deal"]),"BBB",market,AS_OF)
index_cs=price_instrument(json.dumps(inputs["ANALYST-CDX"]),market,AS_OF,model="hazard_rate",metrics=["cs01"])
hedge_units=-bbb.cs01/index_cs.metrics["cs01"]
assert abs(bbb.cs01+hedge_units*index_cs.metrics["cs01"])<1e-8
print({"BBB_zspread_CS01":bbb.cs01,"index_quote_CS01_per_1m":index_cs.metrics["cs01"],
       "formal_index_units":hedge_units})
print("This formal parallel match equates different risk coordinates. It leaves collateral defaults, recovery, cashflow timing, OC diversion, correlation and index/bond basis. Reprice joint scenarios before using it as a trade hedge.")


# Joint stress: market index quotes and collateral assumptions move together.
joint_quotes=tracks.credit_calibration_envelope()
for quote in joint_quotes["market_data"]:
    if quote["kind"]=="cds_quote":quote["spread_bp"]+=100.
joint_market=tracks.build_market("credit")
joint_market.insert(calibrate(joint_quotes).market.get_hazard("CDX-HAZ"))
loss_deal=deepcopy(holding["deal"]);ls=loss_deal["instrument"]["spec"]
ls["default_spec"]={"cdr":.05};ls["recovery_spec"]={"rate":.25,"recovery_lag":6}
ls["default_assumptions"].update({"base_cdr_annual":.05,"base_recovery_rate":.25})
stress_bbb=structured_credit_tranche_metrics(json.dumps(loss_deal),"BBB",joint_market,AS_OF)
stress_index=price_instrument(json.dumps(inputs["ANALYST-CDX"]),joint_market,AS_OF,model="hazard_rate")
residual=stress_bbb.pv-bbb.pv+hedge_units*(stress_index.value.amount-index_cs.value.amount)
assert math.isfinite(residual) and abs(residual)>1.
print({"BBB_stress_pnl":stress_bbb.pv-bbb.pv,
       "index_hedge_pnl":hedge_units*(stress_index.value.amount-index_cs.value.amount),"joint_residual":residual})


### Exercise — Buy protection at the stressed spread

Price a 220 bp payer in the base market and after every calibration quote widens 100 bp. Compare initial cost and scenario gain with the lower-strike payer.

The stressed strike lowers upfront option cost and leaves a larger initial deductible. Its later value is repriced at the same valuation date and expiry. This is a market scenario, not realized carry to the stress date, and it excludes funding and bid/offer.

In [ ]:
stressed_calibration=tracks.credit_calibration_envelope()
for quote in stressed_calibration["market_data"]:
    if quote["kind"]=="cds_quote":quote["spread_bp"]+=100.
stressed_market=tracks.build_market("credit")
stressed_market.insert(calibrate(stressed_calibration).market.get_hazard("CDX-HAZ"))
stressed_strike=deepcopy(payer);stressed_strike["instrument"]["spec"]["strike"]={"spread":"0.022"}
initial_cost=price_instrument(json.dumps(stressed_strike),market,AS_OF,model="bloomberg_cdso").value.amount
stress_value=price_instrument(json.dumps(stressed_strike),stressed_market,AS_OF,model="bloomberg_cdso").value.amount
assert initial_cost<nonko.value.amount
assert stress_value>initial_cost
print({"220bp_strike_cost":initial_cost,"value_after_100bp_parallel_quote_widening":stress_value,
       "scenario_gain_before_financing":stress_value-initial_cost})

### Exercise — Account for a constituent default

After one equally weighted name defaults at 40% recovery, reconcile survivor notional and realized loss. Explain the difference between knockout and non-knockout option settlement.

The factor is applied once to survivor notional; realized loss is stated on original notional. Those quantities are not interchangeable. A settlement ledger must determine whether any front-end loss has already been paid before adding cash to a portfolio report.

In [ ]:
original_notional=1_000_000.;one_name=1/125;recovery=.4
survivor_notional=original_notional*(1-one_name)
settled_loss=original_notional*one_name*(1-recovery)
assert abs(survivor_notional+original_notional*one_name-original_notional)<1e-8
assert abs(settled_loss-original_notional*post_option["instrument"]["spec"]["realized_index_loss"])<1e-8
print({"surviving_index_notional":survivor_notional,"already_realized_loss":settled_loss,
       "post_default_factor":post_option["instrument"]["spec"]["index_factor"]})
print("Exercise delivers protection on the surviving index subject to the option's settlement terms. Non-knockout front-end protection also recognizes relevant loss before expiry; knockout treatment differs. Do not pay an already-settled loss twice or apply the survivor factor twice.")

### Reprice a synthetic tranche along the index quote direction

In [ ]:
one_bp=tracks.credit_calibration_envelope()
for quote in one_bp["market_data"]:
    if quote["kind"]=="cds_quote":quote["spread_bp"]+=1.
bumped_market=tracks.build_market("credit")
bumped_market.insert(calibrate(one_bp).market.get_hazard("CDX-HAZ"))
index_base=price_instrument(json.dumps(inputs["ANALYST-CDX"]),market,AS_OF,model="hazard_rate").value.amount
index_bumped=price_instrument(json.dumps(inputs["ANALYST-CDX"]),bumped_market,AS_OF,model="hazard_rate").value.amount
tranche_base=price_instrument(json.dumps(inputs["CDX-0-3"]),market,AS_OF,model="hazard_rate").value.amount
tranche_bumped=price_instrument(json.dumps(inputs["CDX-0-3"]),bumped_market,AS_OF,model="hazard_rate").value.amount
tranche_delta=(tranche_bumped-tranche_base)/(index_bumped-index_base)
assert math.isfinite(tranche_delta) and index_bumped>index_base
print({"index_1bp_pnl":index_bumped-index_base,"tranche_1bp_pnl":tranche_bumped-tranche_base,
       "tranche_delta_in_index_units":tranche_delta})
print(pd.DataFrame([{"expiry_years":t,"spread_strike_decimal":k,
    "lognormal_spread_vol":market.get_surface("CDX-OPTION-VOL").vol(t,k)}
    for t in [.25,.5,1.] for k in [.0025,.01,.03]]))
